In [0]:
# 1. Setup and Silver Load



# Azure config setup

storage_account_name = "REMOVED"
storage_account_key = "REMOVED"

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",storage_account_key)
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# Define paths
base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"

# Read bronze CSV file

print("Reading bronze CSVs")

df_trans_bronze = spark.read.option("header","true").csv(f"{base_path}/transactions/landing/current.csv")
df_units_bronze = spark.read.option("header","true").csv(f"{base_path}/units/landing/current.csv")
df_proj_bronze = spark.read.option("header","true").option("nullValues","null").csv(f"{base_path}/projects/landing/current.csv")

print("Bronze DataFrame created")



In [0]:
# 2. Clean the projects dimensions
from pyspark.sql.functions import col, to_date, cast

df_proj_clean = df_proj_bronze.select(
    col("project_number").cast("int"),
    col("project_status"),
    col("percent_completed").try_cast("double"),
    to_date(col("project_start_date"),"dd-MM-yyyy").alias("start_date"),
    to_date(col("completion_date"),"dd-MM-yyyy").alias("completion_date"),
    col("master_project_en").alias("master_community")
).dropDuplicates(["project_number"])

print("Project cleaned")
display(df_proj_clean.limit(5))

In [0]:
# 3. Clean Transaction facts
from pyspark.sql.functions import col, to_date, cast
df_trans_clean = df_trans_bronze.select(
    col("transaction_id"),
    to_date(col("instance_date"),"dd-MM-yyyy").alias("transaction_date"),
    col("procedure_name_en").alias("transaction_type"),
    col("trans_group_en").alias("transaction_group"),

    col("area_name_en").alias("area_name"),
    col("building_name_en").alias("building_name"),
    col("project_number").try_cast("int"),

    col("project_name_en").alias("project_name"),
    col("nearest_landmark_en").alias("landmark"),

    col("property_type_en").alias("property_type"),
    col("property_sub_type_en").alias("property_subtype"),
    col("rooms_en").alias("room_type"),
    
    col("procedure_area").cast("double").alias("area_sqft"),
    col("actual_worth").cast("double").alias("price"),
    col("meter_sale_price").cast("double")
)

df_trans_clean = df_trans_clean.filter(col("price")>0)

print("Transaction cleaned")

display(df_trans_clean.limit(5))

In [0]:
# 4. Join and Enrich (Silver master table)

from pyspark.sql.functions import when, col, to_date, when, lit

# Join transactions with project

df_enriched = df_trans_clean.join(
    df_proj_clean,
    on="project_number",
    how="left"
)

# Enrich. Calculate price per sqft

df_enriched = df_enriched.withColumn(
    "price_per_sqft",
    when(col("area_sqft")>0, col("price")/col("area_sqft")).otherwise(0)
)

df_enriched = df_enriched.withColumn(
    "is_offplan_flag",
    when(
        (col("transaction_date")>col("completion_date")) | (col("percent_completed")<100), lit(True)
    ).otherwise(lit(False))
)
    
print("Data Enriched and Joined")
display(df_enriched.select("transaction_date","project_name","room_type","price","is_offplan_flag").limit(10))

In [0]:
# 5. Write to silver layer files
from pyspark.sql.functions import col, to_date, cast
# Write the master transactions table

print("writting transactions to Silver")

df_enriched.write.format("delta").mode("overwrite").save(f"{silver_path}/transactions")

# writing clean Unit table

print("Writting Units to Silver")
df_units_clean = df_units_bronze.select(
    col("property_id"),
    col("project_id").try_cast("int").alias("project_number"),
    col("unit_number"),
    
    col("rooms").alias("bedrooms"),
    col("rooms_en").alias("room_type"),

    col("actual_area").try_cast("double").alias("unit_area")
).dropDuplicates(["property_id"])

df_units_clean.write.format("delta").mode("overwrite").save(f"{silver_path}/units")

print("Silver files written to ADLS")

In [0]:
# 6. Verify the data

from pyspark.sql.functions import count

print("Verifying Silver layer files")

try:
    df_silver_trans = spark.read.format("delta").load(f"{silver_path}/transactions")
    df_silver_units = spark.read.format("delta").load(f"{silver_path}/units")

    count_trans = df_silver_trans.count()
    count_units = df_silver_units.count()

    print(f"Count of Transactions in Silver : {count_trans}")
    print(f"Count of Units in Silver : {count_units}")

    if count_trans > 0:
        print("Trans table is not empty")
    else:
        print("Trans table is empty")
except Exception as e:
    print(f"Could not use Silver table : {e}")
